In [1]:
import os
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

# Thread-safe print lock
print_lock = Lock()

def validate_npz_file(fpath):
    """Validate a single NPZ file and remove if corrupted"""
    fname = os.path.basename(fpath)
    
    # Required keys that must be present in the NPZ file
    required_keys = {
        'keypoints0', 'descriptors0', 'scores0', 'image_size0',
        'keypoints1', 'descriptors1', 'scores1', 'image_size1',
        'matches', 'gt_matches0', 'gt_matches1'
    }
    
    try:
        with np.load(fpath) as data:
            # Check if file can be loaded
            available_keys = set(data.files)
            
            # Check for missing keys
            missing_keys = required_keys - available_keys
            if missing_keys:
                raise ValueError(f"Missing required keys: {missing_keys}")
            
            # Optionally verify data can be accessed (not just keys exist)
            for key in required_keys:
                _ = data[key]
                
        return fname, None, False
    except Exception as e:
        # Remove corrupted file
        try:
            os.remove(fpath)
            return fname, str(e), True
        except Exception as remove_error:
            return fname, f"{str(e)} | Failed to remove: {str(remove_error)}", False

def process_files_multithreaded(data_dir, max_workers=None):
    """
    Process NPZ files using multi-threading
    
    Args:
        data_dir: Directory containing NPZ files
        max_workers: Maximum number of threads (None = use default based on CPU count)
    """
    # Get all NPZ files
    npz_files = [
        os.path.join(data_dir, fname)
        for fname in os.listdir(data_dir)
        if fname.endswith('.npz')
    ]
    
    print(f"Found {len(npz_files)} NPZ files to process")
    print(f"Using {max_workers or 'default'} worker threads\n")
    
    # Process files in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_file = {
            executor.submit(validate_npz_file, fpath): fpath 
            for fpath in npz_files
        }
        
        # Process completed tasks
        completed = 0
        errors = 0
        removed = 0
        
        for future in as_completed(future_to_file):
            fname, error, was_removed = future.result()
            completed += 1
            
            if error:
                errors += 1
                if was_removed:
                    removed += 1
                    with print_lock:
                        print(f'Error with {fname}: {error} [REMOVED]')
                else:
                    with print_lock:
                        print(f'Error with {fname}: {error}')
            
            # Optional: Print progress
            if completed % 100 == 0:
                with print_lock:
                    print(f"Progress: {completed}/{len(npz_files)} files processed")
    
    print(f"\nCompleted! Processed {completed} files with {errors} errors")
    print(f"Removed {removed} corrupted files")

if __name__ == '__main__':
    data_dir = '/data/code/glue-factory/data/finetuning/finetuning_pairs_real/easy'
    
    # You can adjust max_workers based on your needs
    # None = default (usually 5x CPU count for I/O bound tasks)
    # Or specify a number like: max_workers=8
    process_files_multithreaded(data_dir, max_workers=10)

Found 1 NPZ files to process
Using 10 worker threads


Completed! Processed 1 files with 0 errors
Removed 0 corrupted files


In [1]:
"""
Multi-threaded Frame Gap Analyzer for SphereCraft Dataset
Analyzes frame gaps and bins pairs into difficulty categories
"""

import numpy as np
from pathlib import Path
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
import argparse
from tqdm import tqdm
import json
import matplotlib.pyplot as plt


def parse_filename(filename):
    """
    Parse filename like 'berlin_00000006_00000176.npz'
    Returns: (scene_name, frame1, frame2, frame_gap)
    """
    stem = Path(filename).stem
    
    # Pattern: {scene_name}_{frame1}_{frame2}
    pattern = r'(.+?)_(\d+)_(\d+)$'
    match = re.match(pattern, stem)
    
    if not match:
        return None
    
    scene_name = match.group(1)
    frame1 = int(match.group(2))
    frame2 = int(match.group(3))
    frame_gap = abs(frame2 - frame1)
    
    return {
        'filename': filename,
        'scene': scene_name,
        'frame1': frame1,
        'frame2': frame2,
        'gap': frame_gap
    }


def process_file_batch(files):
    """Process a batch of files"""
    results = []
    for file_path in files:
        info = parse_filename(file_path.name)
        if info:
            info['path'] = str(file_path)
            results.append(info)
    return results


def analyze_frame_gaps(data_dir, num_threads=8):
    """
    Analyze frame gaps in dataset using multiple threads
    
    Args:
        data_dir: Path to directory containing .npz files
        num_threads: Number of threads for parallel processing
    """
    print("\n" + "="*70)
    print("FRAME GAP ANALYSIS")
    print("="*70)
    
    data_path = Path(data_dir)
    if not data_path.exists():
        print(f"❌ Error: Directory {data_dir} does not exist!")
        return None
    
    # Get all .npz files
    all_files = list(data_path.glob("*.npz"))
    total_files = len(all_files)
    
    print(f"\n[1] SCANNING DATASET:")
    print(f"  Directory: {data_dir}")
    print(f"  Total .npz files found: {total_files:,}")
    
    if total_files == 0:
        print("❌ No .npz files found!")
        return None
    
    # Split files into batches for threading
    batch_size = max(1, total_files // (num_threads * 4))
    file_batches = [all_files[i:i + batch_size] 
                    for i in range(0, total_files, batch_size)]
    
    print(f"  Using {num_threads} threads with {len(file_batches)} batches")
    
    # Process files in parallel
    all_results = []
    
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = [executor.submit(process_file_batch, batch) 
                   for batch in file_batches]
        
        with tqdm(total=len(futures), desc="Processing batches") as pbar:
            for future in as_completed(futures):
                results = future.result()
                all_results.extend(results)
                pbar.update(1)
    
    print(f"  Successfully parsed: {len(all_results):,}/{total_files:,} files")
    
    if len(all_results) == 0:
        print("❌ No files could be parsed!")
        return None
    
    # Extract frame gaps
    frame_gaps = np.array([r['gap'] for r in all_results])
    
    # Statistics
    print(f"\n[2] FRAME GAP STATISTICS:")
    print(f"  Min gap: {frame_gaps.min()}")
    print(f"  Max gap: {frame_gaps.max()}")
    print(f"  Mean gap: {frame_gaps.mean():.2f}")
    print(f"  Median gap: {np.median(frame_gaps):.2f}")
    print(f"  Std gap: {frame_gaps.std():.2f}")
    
    # Define bins
    # Option 1: Equal-width bins
    bins_config = [
        {'name': 'Very Easy', 'min': 0, 'max': 10, 'color': 'green'},
        {'name': 'Easy', 'min': 10, 'max': 30, 'color': 'lightgreen'},
        {'name': 'Medium', 'min': 30, 'max': 60, 'color': 'yellow'},
        {'name': 'Hard', 'min': 60, 'max': float('inf'), 'color': 'red'},
    ]
    
    # Bin the data
    binned_data = {bin_cfg['name']: [] for bin_cfg in bins_config}
    
    for result in all_results:
        gap = result['gap']
        for bin_cfg in bins_config:
            if bin_cfg['min'] <= gap < bin_cfg['max']:
                binned_data[bin_cfg['name']].append(result)
                break
    
    # Print bin statistics
    print(f"\n[3] BINNED DISTRIBUTION:")
    total = len(all_results)
    
    for bin_cfg in bins_config:
        bin_name = bin_cfg['name']
        count = len(binned_data[bin_name])
        pct = 100 * count / total
        
        if bin_cfg['max'] == float('inf'):
            range_str = f"[{bin_cfg['min']:3d}, ∞)"
        else:
            range_str = f"[{bin_cfg['min']:3d}, {bin_cfg['max']:3d})"
        
        print(f"  {bin_name:12s} {range_str:15s}: {count:7,} pairs ({pct:5.1f}%)")
    
    # Analyze by scene
    print(f"\n[4] TOP SCENES BY PAIR COUNT:")
    scene_counts = defaultdict(int)
    for result in all_results:
        scene_counts[result['scene']] += 1
    
    sorted_scenes = sorted(scene_counts.items(), key=lambda x: x[1], reverse=True)
    for scene, count in sorted_scenes[:10]:
        pct = 100 * count / total
        print(f"  {scene:20s}: {count:6,} pairs ({pct:5.1f}%)")
    
    if len(sorted_scenes) > 10:
        print(f"  ... and {len(sorted_scenes) - 10} more scenes")
    
    # Create visualizations
    print(f"\n[5] CREATING VISUALIZATIONS...")
    create_visualizations(frame_gaps, binned_data, bins_config, 
                         output_dir=Path(data_dir).parent)
    
    # Save binned file lists
    print(f"\n[6] SAVING BINNED FILE LISTS...")
    save_binned_lists(binned_data, bins_config, 
                     output_dir=Path(data_dir).parent)
    
    # Summary statistics for training
    print(f"\n[7] TRAINING RECOMMENDATIONS:")
    
    # Calculate expected losses per bin based on your tests
    expected_losses = {
        'Very Easy': 1.5,
        'Easy': 2.5,
        'Medium': 4.0,
        'Hard': 8.0,
    }
    
    weighted_avg_loss = sum(
        len(binned_data[name]) / total * expected_losses[name]
        for name in expected_losses.keys()
    )
    
    print(f"  Expected average loss (based on distribution):")
    print(f"    Weighted average: ~{weighted_avg_loss:.2f}")
    print(f"\n  Curriculum Learning Strategy:")
    print(f"    Phase 1 (Epochs 0-10): Use 'Very Easy' + 'Easy' bins")
    print(f"      → {len(binned_data['Very Easy']) + len(binned_data['Easy']):,} pairs")
    print(f"      → Expected loss: 1.5 → 1.0")
    print(f"    Phase 2 (Epochs 10-20): Add 'Medium' bin")
    print(f"      → {len(binned_data['Very Easy']) + len(binned_data['Easy']) + len(binned_data['Medium']):,} pairs")
    print(f"      → Expected loss: 2.5 → 1.5")
    print(f"    Phase 3 (Epochs 20-30): Use all data")
    print(f"      → {total:,} pairs")
    print(f"      → Expected loss: 3.5 → 2.5")
    
    print("="*70 + "\n")
    
    return {
        'total_pairs': total,
        'frame_gaps': frame_gaps,
        'binned_data': binned_data,
        'bins_config': bins_config,
        'scene_counts': dict(scene_counts),
        'all_results': all_results,
    }


def create_visualizations(frame_gaps, binned_data, bins_config, output_dir):
    """Create visualization plots"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Histogram of frame gaps
    axes[0, 0].hist(frame_gaps, bins=100, edgecolor='black', alpha=0.7)
    axes[0, 0].set_xlabel('Frame Gap')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('Distribution of Frame Gaps')
    axes[0, 0].grid(alpha=0.3)
    axes[0, 0].set_xlim(0, min(200, frame_gaps.max()))
    
    # Add bin boundaries
    for bin_cfg in bins_config[1:]:
        if bin_cfg['min'] < 200:
            axes[0, 0].axvline(bin_cfg['min'], color='red', 
                              linestyle='--', alpha=0.5)
    
    # 2. Cumulative distribution
    sorted_gaps = np.sort(frame_gaps)
    cumulative = np.arange(1, len(sorted_gaps) + 1) / len(sorted_gaps) * 100
    axes[0, 1].plot(sorted_gaps, cumulative, linewidth=2)
    axes[0, 1].set_xlabel('Frame Gap')
    axes[0, 1].set_ylabel('Cumulative Percentage (%)')
    axes[0, 1].set_title('Cumulative Distribution')
    axes[0, 1].grid(alpha=0.3)
    axes[0, 1].set_xlim(0, min(200, frame_gaps.max()))
    
    # Add bin boundaries
    for bin_cfg in bins_config[1:]:
        if bin_cfg['min'] < 200:
            axes[0, 1].axvline(bin_cfg['min'], color='red', 
                              linestyle='--', alpha=0.5)
    
    # 3. Bin distribution (bar chart)
    bin_names = [cfg['name'] for cfg in bins_config]
    bin_counts = [len(binned_data[name]) for name in bin_names]
    bin_colors = [cfg['color'] for cfg in bins_config]
    
    bars = axes[1, 0].bar(bin_names, bin_counts, color=bin_colors, 
                          edgecolor='black', alpha=0.7)
    axes[1, 0].set_ylabel('Number of Pairs')
    axes[1, 0].set_title('Pairs per Difficulty Bin')
    axes[1, 0].grid(alpha=0.3, axis='y')
    
    # Add count labels on bars
    for bar, count in zip(bars, bin_counts):
        height = bar.get_height()
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                       f'{count:,}',
                       ha='center', va='bottom')
    
    # 4. Expected loss by bin
    expected_losses = {
        'Very Easy': 1.5,
        'Easy': 2.5,
        'Medium': 4.0,
        'Hard': 8.0,
    }
    
    bin_losses = [expected_losses[name] for name in bin_names]
    bars = axes[1, 1].bar(bin_names, bin_losses, color=bin_colors,
                          edgecolor='black', alpha=0.7)
    axes[1, 1].set_ylabel('Expected Loss')
    axes[1, 1].set_title('Expected Loss per Difficulty Bin')
    axes[1, 1].grid(alpha=0.3, axis='y')
    
    # Add loss labels
    for bar, loss in zip(bars, bin_losses):
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{loss:.1f}',
                       ha='center', va='bottom')
    
    plt.tight_layout()
    
    output_path = Path(output_dir) / 'frame_gap_analysis.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"  ✓ Saved visualization to {output_path}")
    plt.close()


def save_binned_lists(binned_data, bins_config, output_dir):
    """Save lists of files for each bin (useful for curriculum learning)"""
    output_dir = Path(output_dir)
    
    for bin_cfg in bins_config:
        bin_name = bin_cfg['name']
        data = binned_data[bin_name]
        
        # Save as JSON
        output_file = output_dir / f"bin_{bin_name.lower().replace(' ', '_')}.json"
        
        bin_info = {
            'bin_name': bin_name,
            'min_gap': bin_cfg['min'],
            'max_gap': bin_cfg['max'] if bin_cfg['max'] != float('inf') else 'inf',
            'count': len(data),
            'files': [
                {
                    'filename': item['filename'],
                    'scene': item['scene'],
                    'frame1': item['frame1'],
                    'frame2': item['frame2'],
                    'gap': item['gap'],
                }
                for item in data
            ]
        }
        
        with open(output_file, 'w') as f:
            json.dump(bin_info, f, indent=2)
        
        print(f"  ✓ Saved {bin_name:12s} bin to {output_file.name} ({len(data):,} pairs)")
    
    # Also save a summary
    summary_file = output_dir / 'binning_summary.json'
    summary = {
        'total_pairs': sum(len(binned_data[cfg['name']]) for cfg in bins_config),
        'bins': [
            {
                'name': cfg['name'],
                'min_gap': cfg['min'],
                'max_gap': cfg['max'] if cfg['max'] != float('inf') else 'inf',
                'count': len(binned_data[cfg['name']]),
                'percentage': 100 * len(binned_data[cfg['name']]) / 
                             sum(len(binned_data[c['name']]) for c in bins_config)
            }
            for cfg in bins_config
        ]
    }
    
    with open(summary_file, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"  ✓ Saved summary to {summary_file.name}")



data_dir = "/data/code/glue-factory/data/finetuning/finetuning_pairs_spherecraft"
threads = 20

# Run analysis
results = analyze_frame_gaps(data_dir, num_threads=threads)

if results:
    print("\n✅ Analysis complete!")
    print(f"\nBinned file lists saved to: {Path(data_dir).parent}")
    print("You can now use these for curriculum learning!")




FRAME GAP ANALYSIS

[1] SCANNING DATASET:
  Directory: /data/code/glue-factory/data/finetuning/finetuning_pairs_spherecraft
  Total .npz files found: 2,552
  Using 20 threads with 83 batches


Processing batches: 100%|██████████| 83/83 [00:00<00:00, 347085.97it/s]

  Successfully parsed: 2,552/2,552 files

[2] FRAME GAP STATISTICS:
  Min gap: 1
  Max gap: 59
  Mean gap: 21.14
  Median gap: 19.00
  Std gap: 13.69

[3] BINNED DISTRIBUTION:
  Very Easy    [  0,  10)     :     645 pairs ( 25.3%)
  Easy         [ 10,  30)     :   1,165 pairs ( 45.7%)
  Medium       [ 30,  60)     :     742 pairs ( 29.1%)
  Hard         [ 60, ∞)       :       0 pairs (  0.0%)

[4] TOP SCENES BY PAIR COUNT:
  barbershop          :  2,552 pairs (100.0%)

[5] CREATING VISUALIZATIONS...


  ✓ Saved visualization to /data/code/glue-factory/data/finetuning/frame_gap_analysis.png

[6] SAVING BINNED FILE LISTS...
  ✓ Saved Very Easy    bin to bin_very_easy.json (645 pairs)
  ✓ Saved Easy         bin to bin_easy.json (1,165 pairs)
  ✓ Saved Medium       bin to bin_medium.json (742 pairs)
  ✓ Saved Hard         bin to bin_hard.json (0 pairs)
  ✓ Saved summary to binning_summary.json

[7] TRAINING RECOMMENDATIONS:
  Expected average loss (based on distribution):
    Weighted average: ~2.68

  Curriculum Learning Strategy:
    Phase 1 (Epochs 0-10): Use 'Very Easy' + 'Easy' bins
      → 1,810 pairs
      → Expected loss: 1.5 → 1.0
    Phase 2 (Epochs 10-20): Add 'Medium' bin
      → 2,552 pairs
      → Expected loss: 2.5 → 1.5
    Phase 3 (Epochs 20-30): Use all data
      → 2,552 pairs
      → Expected loss: 3.5 → 2.5


✅ Analysis complete!

Binned file lists saved to: /data/code/glue-factory/data/finetuning
You can now use these for curriculum learning!


In [1]:
"""
Multi-threaded GT Matches Analyzer for SphereCraft Dataset
Analyzes ground truth match counts and bins pairs into difficulty categories
"""

import numpy as np
from pathlib import Path
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
import argparse
from tqdm import tqdm
import json
import matplotlib.pyplot as plt


def parse_filename(filename):
    """
    Parse filename like 'berlin_00000006_00000176.npz'
    Returns: (scene_name, frame1, frame2)
    """
    stem = Path(filename).stem
    
    # Pattern: {scene_name}_{frame1}_{frame2}
    pattern = r'(.+?)_(\d+)_(\d+)$'
    match = re.match(pattern, stem)
    
    if not match:
        return None
    
    scene_name = match.group(1)
    frame1 = int(match.group(2))
    frame2 = int(match.group(3))
    
    return {
        'scene': scene_name,
        'frame1': frame1,
        'frame2': frame2,
    }


def process_file(file_path):
    """
    Process a single .npz file and extract GT match count
    """
    try:
        # Parse filename
        info = parse_filename(file_path.name)
        if not info:
            return None
        
        # Load .npz file
        data = np.load(str(file_path))
        
        # Count valid GT matches (where gt_matches0 >= 0)
        if 'gt_matches0' in data:
            gt_matches = data['gt_matches0']
            num_matches = (gt_matches >= 0).sum()
        elif 'gt_matches1' in data:
            gt_matches = data['gt_matches1']
            num_matches = (gt_matches >= 0).sum()
        else:
            return None
        
        # Also get frame gap for correlation analysis
        frame_gap = abs(info['frame2'] - info['frame1'])
        
        return {
            'filename': file_path.name,
            'path': str(file_path),
            'scene': info['scene'],
            'frame1': info['frame1'],
            'frame2': info['frame2'],
            'frame_gap': frame_gap,
            'num_matches': int(num_matches),
        }
        
    except Exception as e:
        print(f"Error processing {file_path.name}: {e}")
        return None


def process_file_batch(files):
    """Process a batch of files"""
    results = []
    for file_path in files:
        result = process_file(file_path)
        if result:
            results.append(result)
    return results


def analyze_gt_matches(data_dir, num_threads=8):
    """
    Analyze GT match counts in dataset using multiple threads
    
    Args:
        data_dir: Path to directory containing .npz files
        num_threads: Number of threads for parallel processing
    """
    print("\n" + "="*70)
    print("GROUND TRUTH MATCH COUNT ANALYSIS")
    print("="*70)
    
    data_path = Path(data_dir)
    if not data_path.exists():
        print(f"❌ Error: Directory {data_dir} does not exist!")
        return None
    
    # Get all .npz files
    all_files = list(data_path.glob("*.npz"))
    total_files = len(all_files)
    
    print(f"\n[1] SCANNING DATASET:")
    print(f"  Directory: {data_dir}")
    print(f"  Total .npz files found: {total_files:,}")
    
    if total_files == 0:
        print("❌ No .npz files found!")
        return None
    
    # Split files into batches for threading
    batch_size = max(1, total_files // (num_threads * 4))
    file_batches = [all_files[i:i + batch_size] 
                    for i in range(0, total_files, batch_size)]
    
    print(f"  Using {num_threads} threads with {len(file_batches)} batches")
    print(f"  ⚠️  This will load all .npz files - may take a few minutes...")
    
    # Process files in parallel
    all_results = []
    
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = [executor.submit(process_file_batch, batch) 
                   for batch in file_batches]
        
        with tqdm(total=len(futures), desc="Processing batches") as pbar:
            for future in as_completed(futures):
                results = future.result()
                all_results.extend(results)
                pbar.update(1)
    
    print(f"  Successfully processed: {len(all_results):,}/{total_files:,} files")
    
    if len(all_results) == 0:
        print("❌ No files could be processed!")
        return None
    
    # Extract match counts
    match_counts = np.array([r['num_matches'] for r in all_results])
    frame_gaps = np.array([r['frame_gap'] for r in all_results])
    
    # Statistics
    print(f"\n[2] GT MATCH COUNT STATISTICS:")
    print(f"  Min matches: {match_counts.min()}")
    print(f"  Max matches: {match_counts.max()}")
    print(f"  Mean matches: {match_counts.mean():.2f}")
    print(f"  Median matches: {np.median(match_counts):.2f}")
    print(f"  Std matches: {match_counts.std():.2f}")
    
    # Percentiles for bin design
    percentiles = [10, 25, 50, 75, 90, 95]
    print(f"\n  Percentiles:")
    for p in percentiles:
        val = np.percentile(match_counts, p)
        print(f"    {p:2d}th percentile: {val:.0f} matches")
    
    # Correlation analysis
    correlation = np.corrcoef(match_counts, frame_gaps)[0, 1]
    print(f"\n  Correlation between frame gap and match count: {correlation:.3f}")
    if correlation < -0.5:
        print(f"    → Strong negative correlation: larger gap = fewer matches ✓")
    elif correlation < -0.3:
        print(f"    → Moderate negative correlation")
    else:
        print(f"    → Weak correlation")
    
    # Define bins based on percentiles
    # Design bins so each has reasonable number of samples
    bins_config = [
        {'name': 'Very Hard', 'min': 0, 'max': 150, 'color': 'red'},
        {'name': 'Hard', 'min': 150, 'max': 400, 'color': 'orange'},
        {'name': 'Medium', 'min': 400, 'max': 650, 'color': 'yellow'},
        {'name': 'Easy', 'min': 650, 'max': float('inf'), 'color': 'green'},
    ]
    
    # Bin the data
    binned_data = {bin_cfg['name']: [] for bin_cfg in bins_config}
    
    for result in all_results:
        num_matches = result['num_matches']
        for bin_cfg in bins_config:
            if bin_cfg['min'] <= num_matches < bin_cfg['max']:
                binned_data[bin_cfg['name']].append(result)
                break
    
    # Print bin statistics
    print(f"\n[3] BINNED DISTRIBUTION (by GT match count):")
    total = len(all_results)
    
    for bin_cfg in bins_config:
        bin_name = bin_cfg['name']
        count = len(binned_data[bin_name])
        pct = 100 * count / total
        
        if bin_cfg['max'] == float('inf'):
            range_str = f"[{bin_cfg['min']:3d}, ∞)"
        else:
            range_str = f"[{bin_cfg['min']:3d}, {bin_cfg['max']:3d})"
        
        # Calculate average match count and frame gap for this bin
        if count > 0:
            bin_matches = [r['num_matches'] for r in binned_data[bin_name]]
            bin_gaps = [r['frame_gap'] for r in binned_data[bin_name]]
            avg_matches = np.mean(bin_matches)
            avg_gap = np.mean(bin_gaps)
            
            print(f"  {bin_name:12s} {range_str:15s}: {count:7,} pairs ({pct:5.1f}%)")
            print(f"    └─ Avg matches: {avg_matches:6.1f}, Avg frame gap: {avg_gap:6.1f}")
    
    # Analyze by scene
    print(f"\n[4] TOP SCENES BY PAIR COUNT:")
    scene_counts = defaultdict(int)
    scene_avg_matches = defaultdict(list)
    
    for result in all_results:
        scene_counts[result['scene']] += 1
        scene_avg_matches[result['scene']].append(result['num_matches'])
    
    sorted_scenes = sorted(scene_counts.items(), key=lambda x: x[1], reverse=True)
    for scene, count in sorted_scenes[:10]:
        pct = 100 * count / total
        avg_matches = np.mean(scene_avg_matches[scene])
        print(f"  {scene:20s}: {count:6,} pairs ({pct:5.1f}%), "
              f"avg {avg_matches:6.1f} matches")
    
    if len(sorted_scenes) > 10:
        print(f"  ... and {len(sorted_scenes) - 10} more scenes")
    
    # Analyze extreme cases
    print(f"\n[5] EXTREME CASES:")
    
    # Pairs with very few matches
    very_few = [r for r in all_results if r['num_matches'] < 10]
    print(f"  Pairs with < 10 matches: {len(very_few):,} ({100*len(very_few)/total:.1f}%)")
    if len(very_few) > 0:
        print(f"    Example: {very_few[0]['filename']} ({very_few[0]['num_matches']} matches)")
    
    # Pairs with many matches
    many = [r for r in all_results if r['num_matches'] > 800]
    print(f"  Pairs with > 800 matches: {len(many):,} ({100*len(many)/total:.1f}%)")
    if len(many) > 0:
        print(f"    Example: {many[0]['filename']} ({many[0]['num_matches']} matches)")
    
    # Create visualizations
    print(f"\n[6] CREATING VISUALIZATIONS...")
    create_visualizations(match_counts, frame_gaps, binned_data, bins_config, 
                         output_dir=Path(data_dir).parent)
    
    # Save binned file lists
    print(f"\n[7] SAVING BINNED FILE LISTS...")
    save_binned_lists(binned_data, bins_config, 
                     output_dir=Path(data_dir).parent, 
                     suffix='_by_matches')
    
    # Training recommendations
    print(f"\n[8] TRAINING RECOMMENDATIONS:")
    
    # Expected losses based on match count
    # More matches = easier = lower loss
    expected_losses = {
        'Easy': 1.5,        # 400+ matches
        'Medium': 3.0,      # 200-400 matches
        'Hard': 6.0,        # 50-200 matches
        'Very Hard': 12.0,  # <50 matches
    }
    
    weighted_avg_loss = sum(
        len(binned_data[name]) / total * expected_losses[name]
        for name in expected_losses.keys()
    )
    
    print(f"  Expected average loss (based on distribution):")
    print(f"    Weighted average: ~{weighted_avg_loss:.2f}")
    print(f"\n  Curriculum Learning Strategy (by match count):")
    print(f"    Phase 1 (Epochs 0-10): Use 'Easy' + 'Medium' bins")
    print(f"      → {len(binned_data['Easy']) + len(binned_data['Medium']):,} pairs")
    print(f"      → Expected loss: 2.0 → 1.2")
    print(f"    Phase 2 (Epochs 10-20): Add 'Hard' bin")
    print(f"      → {len(binned_data['Easy']) + len(binned_data['Medium']) + len(binned_data['Hard']):,} pairs")
    print(f"      → Expected loss: 3.5 → 2.0")
    print(f"    Phase 3 (Epochs 20-30): Use all data")
    print(f"      → {total:,} pairs")
    print(f"      → Expected loss: 4.5 → 2.5")
    
    print(f"\n  💡 Match-based binning vs Frame-gap binning:")
    print(f"    • Match-based: More direct indicator of difficulty")
    print(f"    • Frame-gap: Proxy for viewpoint change")
    print(f"    • Correlation: {correlation:.3f} - they're related but different!")
    
    print("="*70 + "\n")
    
    return {
        'total_pairs': total,
        'match_counts': match_counts,
        'frame_gaps': frame_gaps,
        'binned_data': binned_data,
        'bins_config': bins_config,
        'scene_counts': dict(scene_counts),
        'all_results': all_results,
        'correlation': correlation,
    }


def create_visualizations(match_counts, frame_gaps, binned_data, bins_config, output_dir):
    """Create visualization plots"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # 1. Histogram of match counts
    axes[0, 0].hist(match_counts, bins=100, edgecolor='black', alpha=0.7)
    axes[0, 0].set_xlabel('Number of GT Matches')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('Distribution of GT Match Counts')
    axes[0, 0].grid(alpha=0.3)
    
    # Add bin boundaries
    for bin_cfg in bins_config[1:]:
        if bin_cfg['min'] < 1000:
            axes[0, 0].axvline(bin_cfg['min'], color='red', 
                              linestyle='--', alpha=0.5, linewidth=2)
    
    # 2. Cumulative distribution
    sorted_matches = np.sort(match_counts)
    cumulative = np.arange(1, len(sorted_matches) + 1) / len(sorted_matches) * 100
    axes[0, 1].plot(sorted_matches, cumulative, linewidth=2)
    axes[0, 1].set_xlabel('Number of GT Matches')
    axes[0, 1].set_ylabel('Cumulative Percentage (%)')
    axes[0, 1].set_title('Cumulative Distribution')
    axes[0, 1].grid(alpha=0.3)
    
    # Add bin boundaries
    for bin_cfg in bins_config[1:]:
        if bin_cfg['min'] < 1000:
            axes[0, 1].axvline(bin_cfg['min'], color='red', 
                              linestyle='--', alpha=0.5, linewidth=2)
    
    # 3. Scatter: Frame gap vs Match count
    # Sample for visualization (too many points otherwise)
    sample_size = min(10000, len(match_counts))
    indices = np.random.choice(len(match_counts), sample_size, replace=False)
    axes[0, 2].scatter(frame_gaps[indices], match_counts[indices], 
                      alpha=0.3, s=10, c='blue')
    axes[0, 2].set_xlabel('Frame Gap')
    axes[0, 2].set_ylabel('Number of GT Matches')
    axes[0, 2].set_title('Frame Gap vs GT Matches (sampled)')
    axes[0, 2].grid(alpha=0.3)
    
    # Add correlation text
    correlation = np.corrcoef(match_counts, frame_gaps)[0, 1]
    axes[0, 2].text(0.05, 0.95, f'Correlation: {correlation:.3f}',
                   transform=axes[0, 2].transAxes,
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                   verticalalignment='top')
    
    # 4. Bin distribution (bar chart)
    bin_names = [cfg['name'] for cfg in bins_config]
    bin_counts = [len(binned_data[name]) for name in bin_names]
    bin_colors = [cfg['color'] for cfg in bins_config]
    
    bars = axes[1, 0].bar(bin_names, bin_counts, color=bin_colors, 
                          edgecolor='black', alpha=0.7)
    axes[1, 0].set_ylabel('Number of Pairs')
    axes[1, 0].set_title('Pairs per Difficulty Bin')
    axes[1, 0].grid(alpha=0.3, axis='y')
    axes[1, 0].tick_params(axis='x', rotation=15)
    
    # Add count labels on bars
    for bar, count in zip(bars, bin_counts):
        height = bar.get_height()
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                       f'{count:,}',
                       ha='center', va='bottom', fontsize=9)
    
    # 5. Expected loss by bin
    expected_losses = {
        'Easy': 1.5,
        'Medium': 3.0,
        'Hard': 6.0,
        'Very Hard': 12.0,
    }
    
    bin_losses = [expected_losses[name] for name in bin_names]
    bars = axes[1, 1].bar(bin_names, bin_losses, color=bin_colors,
                          edgecolor='black', alpha=0.7)
    axes[1, 1].set_ylabel('Expected Loss')
    axes[1, 1].set_title('Expected Loss per Difficulty Bin')
    axes[1, 1].grid(alpha=0.3, axis='y')
    axes[1, 1].tick_params(axis='x', rotation=15)
    
    # Add loss labels
    for bar, loss in zip(bars, bin_losses):
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{loss:.1f}',
                       ha='center', va='bottom')
    
    # 6. Average matches per bin
    bin_avg_matches = []
    for bin_name in bin_names:
        if len(binned_data[bin_name]) > 0:
            matches = [r['num_matches'] for r in binned_data[bin_name]]
            bin_avg_matches.append(np.mean(matches))
        else:
            bin_avg_matches.append(0)
    
    bars = axes[1, 2].bar(bin_names, bin_avg_matches, color=bin_colors,
                          edgecolor='black', alpha=0.7)
    axes[1, 2].set_ylabel('Average GT Matches')
    axes[1, 2].set_title('Average Matches per Bin')
    axes[1, 2].grid(alpha=0.3, axis='y')
    axes[1, 2].tick_params(axis='x', rotation=15)
    
    # Add labels
    for bar, avg in zip(bars, bin_avg_matches):
        height = bar.get_height()
        axes[1, 2].text(bar.get_x() + bar.get_width()/2., height,
                       f'{avg:.0f}',
                       ha='center', va='bottom')
    
    plt.tight_layout()
    
    output_path = Path(output_dir) / 'gt_match_analysis.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"  ✓ Saved visualization to {output_path}")
    plt.close()


def save_binned_lists(binned_data, bins_config, output_dir, suffix=''):
    """Save lists of files for each bin"""
    output_dir = Path(output_dir)
    
    for bin_cfg in bins_config:
        bin_name = bin_cfg['name']
        data = binned_data[bin_name]
        
        # Save as JSON
        filename = f"bin_{bin_name.lower().replace(' ', '_')}{suffix}.json"
        output_file = output_dir / filename
        
        bin_info = {
            'bin_name': bin_name,
            'min_matches': bin_cfg['min'],
            'max_matches': bin_cfg['max'] if bin_cfg['max'] != float('inf') else 'inf',
            'count': len(data),
            'files': [
                {
                    'filename': item['filename'],
                    'scene': item['scene'],
                    'frame1': item['frame1'],
                    'frame2': item['frame2'],
                    'frame_gap': item['frame_gap'],
                    'num_matches': item['num_matches'],
                }
                for item in data
            ]
        }
        
        with open(output_file, 'w') as f:
            json.dump(bin_info, f, indent=2)
        
        print(f"  ✓ Saved {bin_name:12s} bin to {output_file.name} ({len(data):,} pairs)")
    
    # Also save a summary
    summary_file = output_dir / f'binning_summary{suffix}.json'
    summary = {
        'binning_method': 'gt_match_count',
        'total_pairs': sum(len(binned_data[cfg['name']]) for cfg in bins_config),
        'bins': [
            {
                'name': cfg['name'],
                'min_matches': cfg['min'],
                'max_matches': cfg['max'] if cfg['max'] != float('inf') else 'inf',
                'count': len(binned_data[cfg['name']]),
                'percentage': 100 * len(binned_data[cfg['name']]) / 
                             sum(len(binned_data[c['name']]) for c in bins_config)
            }
            for cfg in bins_config
        ]
    }
    
    with open(summary_file, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"  ✓ Saved summary to {summary_file.name}")


data_dir = "/data/code/glue-factory/data/finetuning/finetuning_pairs_spherecraft"
threads = 20

# Run analysis
results = analyze_gt_matches(data_dir, num_threads=threads)

if results:
    print("\n✅ Analysis complete!")
    print(f"\nBinned file lists saved to: {Path(data_dir).parent}")
    print("Files have '_by_matches' suffix to distinguish from frame-gap bins")
    print("\n💡 You can now use these for curriculum learning based on match count!")


GROUND TRUTH MATCH COUNT ANALYSIS

[1] SCANNING DATASET:
  Directory: /data/code/glue-factory/data/finetuning/finetuning_pairs_spherecraft
  Total .npz files found: 4
  Using 20 threads with 4 batches
  ⚠️  This will load all .npz files - may take a few minutes...


Processing batches: 100%|██████████| 4/4 [00:00<00:00, 83055.52it/s]

  Successfully processed: 4/4 files

[2] GT MATCH COUNT STATISTICS:
  Min matches: 392
  Max matches: 806
  Mean matches: 552.25
  Median matches: 505.50
  Std matches: 158.41

  Percentiles:
    10th percentile: 410 matches
    25th percentile: 436 matches
    50th percentile: 506 matches
    75th percentile: 622 matches
    90th percentile: 732 matches
    95th percentile: 769 matches

  Correlation between frame gap and match count: -0.998
    → Strong negative correlation: larger gap = fewer matches ✓

[3] BINNED DISTRIBUTION (by GT match count):
  Hard         [150, 400)     :       1 pairs ( 25.0%)
    └─ Avg matches:  392.0, Avg frame gap:   45.0
  Medium       [400, 650)     :       2 pairs ( 50.0%)
    └─ Avg matches:  505.5, Avg frame gap:   31.0
  Easy         [650, ∞)       :       1 pairs ( 25.0%)
    └─ Avg matches:  806.0, Avg frame gap:    1.0

[4] TOP SCENES BY PAIR COUNT:
  barbershop          :      4 pairs (100.0%), avg  552.2 matches

[5] EXTREME CASES:
  Pairs wit

  ✓ Saved visualization to /data/code/glue-factory/data/finetuning/gt_match_analysis.png

[7] SAVING BINNED FILE LISTS...
  ✓ Saved Very Hard    bin to bin_very_hard_by_matches.json (0 pairs)
  ✓ Saved Hard         bin to bin_hard_by_matches.json (1 pairs)
  ✓ Saved Medium       bin to bin_medium_by_matches.json (2 pairs)
  ✓ Saved Easy         bin to bin_easy_by_matches.json (1 pairs)
  ✓ Saved summary to binning_summary_by_matches.json

[8] TRAINING RECOMMENDATIONS:
  Expected average loss (based on distribution):
    Weighted average: ~3.38

  Curriculum Learning Strategy (by match count):
    Phase 1 (Epochs 0-10): Use 'Easy' + 'Medium' bins
      → 3 pairs
      → Expected loss: 2.0 → 1.2
    Phase 2 (Epochs 10-20): Add 'Hard' bin
      → 4 pairs
      → Expected loss: 3.5 → 2.0
    Phase 3 (Epochs 20-30): Use all data
      → 4 pairs
      → Expected loss: 4.5 → 2.5

  💡 Match-based binning vs Frame-gap binning:
    • Match-based: More direct indicator of difficulty
    • Frame-g